# 에이전트와 도구 — Agents and Tools

**Skilljar Lessons 05-06 대응**

이 노트북에서 다루는 내용:
1. 에이전틱 루프 (Agentic Loop) 구현
2. 도구 정의 및 실행
3. 환경 인스펙션 (Environment Inspection)
4. `InspectableAgent` 클래스

In [ ]:
# ── Setup ──────────────────────────────────────────────
import anthropic
import json
import time
import os
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

## §1. 도구 정의

에이전트가 사용할 도구(파일 읽기, 디렉토리 목록, 파일 쓰기)를 정의합니다.

In [ ]:
tools = [
    {
        "name": "read_file",
        "description": "Read the contents of a file at the given path.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "The file path to read"}
            },
            "required": ["path"]
        }
    },
    {
        "name": "list_directory",
        "description": "List all files and directories at the given path.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "The directory path to list"}
            },
            "required": ["path"]
        }
    },
    {
        "name": "write_file",
        "description": "Write content to a file at the given path.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "The file path to write to"},
                "content": {"type": "string", "description": "The content to write"}
            },
            "required": ["path", "content"]
        }
    }
]


def execute_tool(name, input_data):
    """도구 이름에 따라 실제 함수를 실행한다."""
    if name == "read_file":
        try:
            with open(input_data["path"], "r") as f:
                return f.read()
        except FileNotFoundError:
            return f"Error: File not found: {input_data['path']}"
    elif name == "list_directory":
        try:
            items = os.listdir(input_data["path"])
            return "\n".join(items)
        except FileNotFoundError:
            return f"Error: Directory not found: {input_data['path']}"
    elif name == "write_file":
        with open(input_data["path"], "w") as f:
            f.write(input_data["content"])
        return f"Successfully wrote to {input_data['path']}"
    return f"Unknown tool: {name}"

print(f"도구 {len(tools)}개 정의 완료: {[t['name'] for t in tools]}")

## §2. 에이전틱 루프 구현

`stop_reason`이 `"tool_use"`인 동안 계속 루프를 돌며 도구를 실행합니다.  
`stop_reason`이 `"end_turn"`이 되면 LLM이 작업 완료를 판단한 것입니다.

In [ ]:
def run_agent(user_message, max_turns=10):
    """에이전틱 루프를 실행한다."""
    messages = [{"role": "user", "content": user_message}]
    turn = 0
    
    print(f"👤 사용자: {user_message}\n")
    
    while turn < max_turns:
        turn += 1
        print(f"--- Turn {turn} ---")
        
        response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            system=("You are a helpful file management agent. "
                    "Use the available tools to complete the user's request. "
                    "Think step by step."),
            tools=tools,
            messages=messages
        )
        
        messages.append({"role": "assistant", "content": response.content})
        
        if response.stop_reason == "end_turn":
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text += block.text
            print(f"🤖 에이전트: {final_text}")
            return final_text
        
        elif response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 {block.name}({json.dumps(block.input, ensure_ascii=False)})")
                    result = execute_tool(block.name, block.input)
                    preview = str(result)[:100]
                    print(f"     -> {preview}{'...' if len(str(result)) > 100 else ''}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(result)
                    })
            
            messages.append({"role": "user", "content": tool_results})
    
    return "Max turns reached."

In [ ]:
# 테스트: 현재 디렉토리의 파일 목록 확인
result = run_agent("List the files in the current directory and tell me what you see.")

## §3. 환경 인스펙션 (InspectableAgent)

에이전트의 **모든 행동과 결정을 로깅**하는 클래스입니다.  
추론 과정, 도구 호출, 토큰 사용량, 비용을 추적합니다.

In [ ]:
class InspectableAgent:
    """관찰 가능한 에이전트 — 모든 행동과 결정을 로깅한다."""
    
    def __init__(self, model=MODEL, agent_tools=None):
        self.client = anthropic.Anthropic()
        self.model = model
        self.tools = agent_tools or tools
        self.trace = []
        self.total_input_tokens = 0
        self.total_output_tokens = 0
    
    def _extract_reasoning(self, response):
        texts = []
        for block in response.content:
            if hasattr(block, "text"):
                texts.append(block.text)
        return " ".join(texts) if texts else "(no reasoning)"
    
    def _log_turn(self, turn_num, response, tool_calls):
        entry = {
            "turn": turn_num,
            "timestamp": time.time(),
            "stop_reason": response.stop_reason,
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "tool_calls": tool_calls,
            "reasoning": self._extract_reasoning(response)
        }
        self.trace.append(entry)
        self.total_input_tokens += response.usage.input_tokens
        self.total_output_tokens += response.usage.output_tokens
    
    def run(self, user_message, max_turns=10):
        """인스펙션이 포함된 에이전틱 루프"""
        messages = [{"role": "user", "content": user_message}]
        turn = 0
        
        while turn < max_turns:
            turn += 1
            response = self.client.messages.create(
                model=self.model,
                max_tokens=4096,
                system="You are a helpful agent. Use tools to complete tasks.",
                tools=self.tools,
                messages=messages
            )
            messages.append({"role": "assistant", "content": response.content})
            
            if response.stop_reason == "end_turn":
                self._log_turn(turn, response, [])
                return self._extract_reasoning(response)
            
            elif response.stop_reason == "tool_use":
                tool_calls = []
                tool_results = []
                for block in response.content:
                    if block.type == "tool_use":
                        result = execute_tool(block.name, block.input)
                        tool_calls.append({
                            "name": block.name,
                            "input": block.input,
                            "result_preview": str(result)[:200]
                        })
                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": str(result)
                        })
                
                self._log_turn(turn, response, tool_calls)
                messages.append({"role": "user", "content": tool_results})
        
        return "Max turns reached."
    
    def print_trace(self):
        """실행 추적 로그를 출력한다."""
        print("=" * 60)
        print("🔍 에이전트 실행 추적 (Execution Trace)")
        print("=" * 60)
        
        for entry in self.trace:
            print(f"\n--- Turn {entry['turn']} ---")
            print(f"  Stop reason: {entry['stop_reason']}")
            print(f"  Tokens: {entry['input_tokens']} in / "
                  f"{entry['output_tokens']} out")
            
            if entry["reasoning"] != "(no reasoning)":
                print(f"  Reasoning: {entry['reasoning'][:120]}...")
            
            for tc in entry["tool_calls"]:
                print(f"  Tool: {tc['name']}")
                print(f"    Input: {json.dumps(tc['input'], ensure_ascii=False)}")
                print(f"    Result: {tc['result_preview'][:80]}...")
        
        print(f"\n{'=' * 60}")
        print(f"📊 Total tokens: {self.total_input_tokens} in / "
              f"{self.total_output_tokens} out")
        print(f"📊 Total turns: {len(self.trace)}")
        cost = (self.total_input_tokens * 0.80 + self.total_output_tokens * 4.0) / 1_000_000
        print(f"💰 Estimated cost: ${cost:.4f}")

In [ ]:
# InspectableAgent 실행
agent = InspectableAgent()
result = agent.run("List the files in the current directory.")

print("\n🤖 최종 응답:")
print(result)

# 실행 추적 확인
agent.print_trace()

## §4. 핵심 정리

- **에이전틱 루프**: `stop_reason == "tool_use"` → 도구 실행 → 반복
- **자율적 결정**: LLM이 어떤 도구를, 어떤 순서로, 몇 번 호출할지 결정
- **`max_turns` 안전장치**: 무한 루프 방지 및 비용 제어
- **환경 인스펙션**: 추론·도구·비용 로깅으로 관찰 가능성 확보
- **다음 노트북**: `S8_06_practice.ipynb`에서 직접 실습